# Data Analysis Bootcamp — Lesson 12
## Advanced GroupBy, `transform()`, and Window-Style Analysis

### Learning Objectives

By the end of this lesson, you will be able to:

- Understand the difference between `groupby()` and `transform()`
- Calculate group-level statistics while preserving row-level data
- Compare individual records against group averages
- Calculate group-level percentages and totals
- Perform group-wise ranking
- Calculate differences and percentage differences from group benchmarks
- Use `shift()`, `cumsum()`, `cumcount()`, and `rolling()`
- Perform window-style analysis
- Apply these techniques to manufacturing and QA data

### Core Idea

> **Aggregation shrinks the data. Transformation preserves the original row structure.**

We will use the Seaborn `tips` dataset for the main examples and a small manufacturing/QA dataset for practical application.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")

tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 1. Inspect the Dataset

Before analysis, inspect the structure and data types.

In [2]:
tips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [3]:
tips.describe(include="all")

,total_bill,tip,sex,smoker,day,time,size
count,244.000000,244.000000,244,244,244,244,244.000000
unique,NaN,NaN,2,2,4,2,NaN
top,NaN,NaN,Male,No,Sat,Dinner,NaN
freq,NaN,NaN,157,151,87,176,NaN
mean,19.785943,2.998279,NaN,NaN,NaN,NaN,2.569672
std,8.902412,1.383638,NaN,NaN,NaN,NaN,0.951100
min,3.070000,1.000000,NaN,NaN,NaN,NaN,1.000000
25%,13.347500,2.000000,NaN,NaN,NaN,NaN,2.000000
50%,17.795000,2.900000,NaN,NaN,NaN,NaN,2.000000
75%,24.127500,3.562500,NaN,NaN,NaN,NaN,3.000000


## 2. Review: Normal `groupby()`

Suppose we want the average bill for each day.

In [4]:
day_average = tips.groupby("day")["total_bill"].mean()
day_average

/tmp/ipykernel_16/336808914.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  day_average = tips.groupby("day")["total_bill"].mean()


day
Thur    17.682742
Fri     17.151579
Sat     20.441379
Sun     21.410000
Name: total_bill, dtype: float64

The grouped result contains **one row per group**.

The original dataset has many transaction rows, while the grouped result has only one result for each day.

But what if we want the average for the relevant day attached to **every original transaction**?

That is where `transform()` becomes important.

# 3. `transform()` — Group Statistics on Every Original Row

`groupby().transform()` performs a group-level calculation but returns a Series aligned with the original DataFrame.

This allows us to compare each individual record against its group.

In [5]:
tips["day_avg_bill"] = (
    tips.groupby("day")["total_bill"]
    .transform("mean")
)

tips[["day", "total_bill", "day_avg_bill"]].head(15)

/tmp/ipykernel_16/2626904496.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]


,day,total_bill,day_avg_bill
0,Sun,16.99,21.41
1,Sun,10.34,21.41
2,Sun,21.01,21.41
3,Sun,23.68,21.41
4,Sun,24.59,21.41
5,Sun,25.29,21.41
6,Sun,8.77,21.41
7,Sun,26.88,21.41
8,Sun,15.04,21.41
9,Sun,14.78,21.41


Every transaction receives the average bill for its corresponding day.

For example:

- A Sunday transaction receives the Sunday average.
- A Saturday transaction receives the Saturday average.
- A Thursday transaction receives the Thursday average.

## 4. Compare Each Transaction with Its Day Average

In [6]:
tips["above_day_average"] = np.where(
    tips["total_bill"] > tips["day_avg_bill"],
    "Yes",
    "No"
)

tips[
    ["day", "total_bill", "day_avg_bill", "above_day_average"]
].head(15)

,day,total_bill,day_avg_bill,above_day_average
0,Sun,16.99,21.41,No
1,Sun,10.34,21.41,No
2,Sun,21.01,21.41,No
3,Sun,23.68,21.41,Yes
4,Sun,24.59,21.41,Yes
5,Sun,25.29,21.41,Yes
6,Sun,8.77,21.41,No
7,Sun,26.88,21.41,Yes
8,Sun,15.04,21.41,No
9,Sun,14.78,21.41,No


### Business Question

> Was this transaction above the typical transaction value for that day?

This is a much more useful analytical feature than simply knowing the transaction amount.

## 5. Difference from Group Average

Calculate the absolute difference between the transaction and its day average.

In [7]:
tips["difference_from_day_avg"] = (
    tips["total_bill"] - tips["day_avg_bill"]
).round(2)

tips[
    ["day", "total_bill", "day_avg_bill", "difference_from_day_avg"]
].head(10)

,day,total_bill,day_avg_bill,difference_from_day_avg
0,Sun,16.99,21.41,-4.42
1,Sun,10.34,21.41,-11.07
2,Sun,21.01,21.41,-0.40
3,Sun,23.68,21.41,2.27
4,Sun,24.59,21.41,3.18
5,Sun,25.29,21.41,3.88
6,Sun,8.77,21.41,-12.64
7,Sun,26.88,21.41,5.47
8,Sun,15.04,21.41,-6.37
9,Sun,14.78,21.41,-6.63


Interpretation:

- Positive value → above the day average
- Negative value → below the day average
- Zero → equal to the day average

## 6. Percentage Difference from Group Average

Formula:

\[
Percentage\ Difference =

rac{Actual - GroupAverage}{GroupAverage}	imes100
\]

In [8]:
tips["pct_diff_from_day_avg"] = (
    (tips["total_bill"] - tips["day_avg_bill"])
    / tips["day_avg_bill"]
    * 100
).round(2)

tips[
    [
        "day",
        "total_bill",
        "day_avg_bill",
        "pct_diff_from_day_avg"
    ]
].head(10)

,day,total_bill,day_avg_bill,pct_diff_from_day_avg
0,Sun,16.99,21.41,-20.64
1,Sun,10.34,21.41,-51.70
2,Sun,21.01,21.41,-1.87
3,Sun,23.68,21.41,10.60
4,Sun,24.59,21.41,14.85
5,Sun,25.29,21.41,18.12
6,Sun,8.77,21.41,-59.04
7,Sun,26.88,21.41,25.55
8,Sun,15.04,21.41,-29.75
9,Sun,14.78,21.41,-30.97


If a transaction is 30 and the group average is 20:

\[

rac{30-20}{20}	imes100 = 50\%
\]

So the transaction is 50% above the group average.

# 7. `transform("sum")`

We can also attach the total revenue for each day to every transaction.

In [9]:
tips["day_total_bill"] = (
    tips.groupby("day")["total_bill"]
    .transform("sum")
)

tips[
    ["day", "total_bill", "day_total_bill"]
].head(10)

/tmp/ipykernel_16/2456681931.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]


,day,total_bill,day_total_bill
0,Sun,16.99,1627.16
1,Sun,10.34,1627.16
2,Sun,21.01,1627.16
3,Sun,23.68,1627.16
4,Sun,24.59,1627.16
5,Sun,25.29,1627.16
6,Sun,8.77,1627.16
7,Sun,26.88,1627.16
8,Sun,15.04,1627.16
9,Sun,14.78,1627.16


## 8. Percentage Contribution Within Group

Now calculate the percentage of the day's total revenue contributed by each transaction.

In [10]:
tips["pct_of_day_revenue"] = (
    tips["total_bill"]
    / tips["day_total_bill"]
    * 100
).round(2)

tips[
    [
        "day",
        "total_bill",
        "day_total_bill",
        "pct_of_day_revenue"
    ]
].head(10)

,day,total_bill,day_total_bill,pct_of_day_revenue
0,Sun,16.99,1627.16,1.04
1,Sun,10.34,1627.16,0.64
2,Sun,21.01,1627.16,1.29
3,Sun,23.68,1627.16,1.46
4,Sun,24.59,1627.16,1.51
5,Sun,25.29,1627.16,1.55
6,Sun,8.77,1627.16,0.54
7,Sun,26.88,1627.16,1.65
8,Sun,15.04,1627.16,0.92
9,Sun,14.78,1627.16,0.91


This pattern is common in business analysis:

\[
Individual\ Value \div Group\ Total 	imes100
\]

It can be used for:

- Revenue contribution
- Product contribution
- Defect contribution
- Customer contribution
- Production contribution

# 9. Group-Level Percentage Using Boolean Data

Suppose we want to know the percentage of transactions involving smokers for each day.

First convert the condition into 1/0.

In [11]:
tips["smoker_flag"] = np.where(
    tips["smoker"] == "Yes",
    1,
    0
)

tips[["day", "smoker", "smoker_flag"]].head(10)

,day,smoker,smoker_flag
0,Sun,No,0
1,Sun,No,0
2,Sun,No,0
3,Sun,No,0
4,Sun,No,0
5,Sun,No,0
6,Sun,No,0
7,Sun,No,0
8,Sun,No,0
9,Sun,No,0


In [12]:
tips["smoker_pct_day"] = (
    tips.groupby("day")["smoker_flag"]
    .transform("mean")
    * 100
).round(2)

tips[
    ["day", "smoker", "smoker_pct_day"]
].head(15)

/tmp/ipykernel_16/4071683505.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["smoker_flag"]


,day,smoker,smoker_pct_day
0,Sun,No,25.0
1,Sun,No,25.0
2,Sun,No,25.0
3,Sun,No,25.0
4,Sun,No,25.0
5,Sun,No,25.0
6,Sun,No,25.0
7,Sun,No,25.0
8,Sun,No,25.0
9,Sun,No,25.0


Why does `mean()` work?

If:

- Yes → 1
- No → 0

then:

\[
Mean = 
rac{Number\ of\ Yes}{Total\ Records}
\]

Multiplying by 100 gives the percentage.

# 10. Group-wise Ranking

Now rank transactions separately within each day.

In [13]:
tips["day_rank"] = (
    tips.groupby("day")["total_bill"]
    .rank(
        ascending=False,
        method="dense"
    )
)

tips[
    ["day", "total_bill", "day_rank"]
].sort_values(
    ["day", "day_rank"]
).head(20)

/tmp/ipykernel_16/3894858113.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]


,day,total_bill,day_rank
197,Thur,43.11,1.0
142,Thur,41.19,2.0
85,Thur,34.83,3.0
141,Thur,34.30,4.0
83,Thur,32.68,5.0
125,Thur,29.80,6.0
192,Thur,28.44,7.0
77,Thur,27.20,8.0
143,Thur,27.05,9.0
88,Thur,24.71,10.0


Rank 1 represents the highest bill within that day.

This is different from a global ranking because each group has its own ranking.

## 11. Top Transaction Within Each Day

In [14]:
top_transactions = tips[
    tips["day_rank"] == 1
][
    ["day", "total_bill", "tip", "size"]
].sort_values("day")

top_transactions

,day,total_bill,tip,size
197,Thur,43.11,5.00,4
95,Fri,40.17,4.73,4
170,Sat,50.81,10.00,3
156,Sun,48.17,5.00,6


This answers:

> What was the highest-value transaction on each day?

# 12. Group-wise Standardization

Sometimes different groups have different averages and variability.

We can calculate a group-specific z-score:

\[
Z = 
rac{x-\mu_{group}}{\sigma_{group}}
\]

In [15]:
tips["day_mean"] = (
    tips.groupby("day")["total_bill"]
    .transform("mean")
)

tips["day_std"] = (
    tips.groupby("day")["total_bill"]
    .transform("std")
)

tips["day_zscore"] = (
    (tips["total_bill"] - tips["day_mean"])
    / tips["day_std"]
)

tips[
    [
        "day",
        "total_bill",
        "day_mean",
        "day_std",
        "day_zscore"
    ]
].head(10)

/tmp/ipykernel_16/3602641917.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]
/tmp/ipykernel_16/3602641917.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]


,day,total_bill,day_mean,day_std,day_zscore
0,Sun,16.99,21.41,8.832122,-0.500446
1,Sun,10.34,21.41,8.832122,-1.253379
2,Sun,21.01,21.41,8.832122,-0.045289
3,Sun,23.68,21.41,8.832122,0.257016
4,Sun,24.59,21.41,8.832122,0.360049
5,Sun,25.29,21.41,8.832122,0.439306
6,Sun,8.77,21.41,8.832122,-1.431140
7,Sun,26.88,21.41,8.832122,0.619330
8,Sun,15.04,21.41,8.832122,-0.721231
9,Sun,14.78,21.41,8.832122,-0.750669


This answers:

> How unusual is this transaction relative to other transactions on the same day?

This idea is particularly useful when groups have different baselines.

# 13. `agg()` vs `transform()`

This is one of the most important concepts in the lesson.

### Aggregation

```python
tips.groupby("day")["total_bill"].mean()
```

Produces one result per group.

### Transformation

```python
tips.groupby("day")["total_bill"].transform("mean")
```

Produces one result for every original row.

| Operation | Result |
|---|---|
| `groupby().agg()` | Summary with fewer rows |
| `groupby().transform()` | Group statistic aligned to original rows |

### Memory Trick

> **Aggregation shrinks. Transformation preserves the row structure.**

# 14. Multiple Group Statistics with `transform()`

In [16]:
tips["day_mean"] = (
    tips.groupby("day")["total_bill"]
    .transform("mean")
)

tips["day_median"] = (
    tips.groupby("day")["total_bill"]
    .transform("median")
)

tips["day_max"] = (
    tips.groupby("day")["total_bill"]
    .transform("max")
)

tips["day_min"] = (
    tips.groupby("day")["total_bill"]
    .transform("min")
)

tips[
    [
        "day",
        "total_bill",
        "day_mean",
        "day_median",
        "day_min",
        "day_max"
    ]
].head(10)

/tmp/ipykernel_16/2163385487.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]
/tmp/ipykernel_16/2163385487.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]
/tmp/ipykernel_16/2163385487.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["total_bill"]
/tmp/ipykernel_16/2163385487.py:17: FutureWarning: The default of observed=Fal

,day,total_bill,day_mean,day_median,day_min,day_max
0,Sun,16.99,21.41,19.63,7.25,48.17
1,Sun,10.34,21.41,19.63,7.25,48.17
2,Sun,21.01,21.41,19.63,7.25,48.17
3,Sun,23.68,21.41,19.63,7.25,48.17
4,Sun,24.59,21.41,19.63,7.25,48.17
5,Sun,25.29,21.41,19.63,7.25,48.17
6,Sun,8.77,21.41,19.63,7.25,48.17
7,Sun,26.88,21.41,19.63,7.25,48.17
8,Sun,15.04,21.41,19.63,7.25,48.17
9,Sun,14.78,21.41,19.63,7.25,48.17


# 15. Relative-to-Group Mean

Another useful feature is the ratio of the observation to its group average.

\[
Relative\ Value = 
rac{x}{GroupMean}
\]

In [17]:
tips["relative_to_day_mean"] = (
    tips["total_bill"] / tips["day_mean"]
).round(3)

tips[
    ["day", "total_bill", "day_mean", "relative_to_day_mean"]
].head(10)

,day,total_bill,day_mean,relative_to_day_mean
0,Sun,16.99,21.41,0.794
1,Sun,10.34,21.41,0.483
2,Sun,21.01,21.41,0.981
3,Sun,23.68,21.41,1.106
4,Sun,24.59,21.41,1.149
5,Sun,25.29,21.41,1.181
6,Sun,8.77,21.41,0.410
7,Sun,26.88,21.41,1.255
8,Sun,15.04,21.41,0.702
9,Sun,14.78,21.41,0.690


Interpretation:

- `1.00` → exactly at group average
- `1.20` → 20% above group average
- `0.80` → 20% below group average

# 16. Window-Style Analysis

Window-style analysis examines a row together with related rows.

Common tools include:

- `shift()` — previous/next observation
- `cumsum()` — running total
- `cumcount()` — sequence number within a group
- `rolling()` — moving calculations
- `rank()` — relative position

# 17. `shift()` — Previous or Next Row

First sort the data.

> Sorting is critical when the row order has analytical meaning.

In [18]:
tips_sorted = tips.sort_values(
    ["day", "total_bill"]
).copy()

tips_sorted["previous_bill"] = (
    tips_sorted["total_bill"].shift(1)
)

tips_sorted[
    ["day", "total_bill", "previous_bill"]
].head(10)

,day,total_bill,previous_bill
149,Thur,7.51,NaN
195,Thur,7.56,7.51
145,Thur,8.35,7.56
135,Thur,8.51,8.35
126,Thur,8.52,8.51
148,Thur,9.78,8.52
82,Thur,10.07,9.78
136,Thur,10.33,10.07
196,Thur,10.34,10.33
117,Thur,10.65,10.34


`shift(1)` moves the previous row's value into the current row.

In time-series analysis, always sort chronologically before using `shift()`.

# 18. Difference from Previous Value

In [19]:
tips_sorted["bill_change"] = (
    tips_sorted["total_bill"]
    - tips_sorted["previous_bill"]
).round(2)

tips_sorted[
    ["day", "total_bill", "previous_bill", "bill_change"]
].head(10)

,day,total_bill,previous_bill,bill_change
149,Thur,7.51,NaN,NaN
195,Thur,7.56,7.51,0.05
145,Thur,8.35,7.56,0.79
135,Thur,8.51,8.35,0.16
126,Thur,8.52,8.51,0.01
148,Thur,9.78,8.52,1.26
82,Thur,10.07,9.78,0.29
136,Thur,10.33,10.07,0.26
196,Thur,10.34,10.33,0.01
117,Thur,10.65,10.34,0.31


## Percentage Change

In [20]:
tips_sorted["bill_pct_change"] = (
    tips_sorted["total_bill"]
    .pct_change()
    * 100
).round(2)

tips_sorted[
    ["total_bill", "previous_bill", "bill_pct_change"]
].head(10)

,total_bill,previous_bill,bill_pct_change
149,7.51,NaN,NaN
195,7.56,7.51,0.67
145,8.35,7.56,10.45
135,8.51,8.35,1.92
126,8.52,8.51,0.12
148,9.78,8.52,14.79
82,10.07,9.78,2.97
136,10.33,10.07,2.58
196,10.34,10.33,0.10
117,10.65,10.34,3.00


# 19. `cumsum()` — Running Total

A cumulative sum produces a running total.

In [21]:
tips_sorted["running_total"] = (
    tips_sorted["total_bill"].cumsum()
)

tips_sorted[
    ["total_bill", "running_total"]
].head(10)

,total_bill,running_total
149,7.51,7.51
195,7.56,15.07
145,8.35,23.42
135,8.51,31.93
126,8.52,40.45
148,9.78,50.23
82,10.07,60.30
136,10.33,70.63
196,10.34,80.97
117,10.65,91.62


Conceptually:

```text
Transaction 1 → 20
Transaction 2 → 20 + 15 = 35
Transaction 3 → 35 + 25 = 60
...
```

# 20. Group-wise Cumulative Sum

Now calculate a running total separately for each day.

In [22]:
tips_sorted = tips.sort_values(
    ["day", "total_bill"]
).copy()

tips_sorted["day_running_total"] = (
    tips_sorted
    .groupby("day")["total_bill"]
    .cumsum()
)

tips_sorted[
    ["day", "total_bill", "day_running_total"]
].head(15)

/tmp/ipykernel_16/1024108516.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("day")["total_bill"]


,day,total_bill,day_running_total
149,Thur,7.51,7.51
195,Thur,7.56,15.07
145,Thur,8.35,23.42
135,Thur,8.51,31.93
126,Thur,8.52,40.45
148,Thur,9.78,50.23
82,Thur,10.07,60.30
136,Thur,10.33,70.63
196,Thur,10.34,80.97
117,Thur,10.65,91.62


The cumulative total restarts when the group changes.

# 21. `cumcount()` — Sequence Number Within Group

In [23]:
tips_sorted["transaction_number"] = (
    tips_sorted.groupby("day").cumcount() + 1
)

tips_sorted[
    ["day", "total_bill", "transaction_number"]
].head(15)

/tmp/ipykernel_16/1234332544.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips_sorted.groupby("day").cumcount() + 1


,day,total_bill,transaction_number
149,Thur,7.51,1
195,Thur,7.56,2
145,Thur,8.35,3
135,Thur,8.51,4
126,Thur,8.52,5
148,Thur,9.78,6
82,Thur,10.07,7
136,Thur,10.33,8
196,Thur,10.34,9
117,Thur,10.65,10


This answers:

> Which transaction number was this within the group?

The `+1` makes the first transaction number 1 instead of 0.

# 22. Rolling Average

A rolling average calculates an average over a moving window.

In [24]:
series = pd.Series([100, 110, 120, 130, 140])

rolling_average = series.rolling(window=3).mean()

rolling_average

0      NaN
1      NaN
2    110.0
3    120.0
4    130.0
dtype: float64

For a 3-period rolling average:

- The first two positions don't yet have three observations.
- Once three observations are available, the average moves forward one observation at a time.

# 23. Why Rolling Average Matters in Manufacturing

Suppose daily defect rates are:

```text
2%
3%
2%
4%
7%
8%
9%
```

A rolling average can help reveal a gradual deterioration in process performance even when individual measurements fluctuate.

For time-dependent data:

1. Sort by time.
2. Define an appropriate window.
3. Calculate the rolling statistic.
4. Interpret the result with process knowledge.

# 24. Comparing Different Window Operations

### Question 1
What is the average bill for each day?

```python
groupby().mean()
```

### Question 2
What is the average bill for this transaction's day?

```python
groupby().transform("mean")
```

### Question 3
What was the previous observation?

```python
shift()
```

### Question 4
What is the average of the previous three observations?

```python
rolling(3).mean()
```

These operations answer different analytical questions.

# 25. Manufacturing / QA Example

Let's create a small production dataset.

In [25]:
production = pd.DataFrame({
    "Line": [
        "A", "A", "A", "A",
        "B", "B", "B", "B"
    ],
    "Part": [
        "P1", "P2", "P3", "P4",
        "P5", "P6", "P7", "P8"
    ],
    "Measurement": [
        10.01, 10.04, 9.98, 10.08,
        9.95, 9.97, 10.02, 10.10
    ]
})

production

,Line,Part,Measurement
0,A,P1,10.01
1,A,P2,10.04
2,A,P3,9.98
3,A,P4,10.08
4,B,P5,9.95
5,B,P6,9.97
6,B,P7,10.02
7,B,P8,10.10


## 26. Average Measurement by Production Line

In [26]:
production["line_avg"] = (
    production
    .groupby("Line")["Measurement"]
    .transform("mean")
)

production

,Line,Part,Measurement,line_avg
0,A,P1,10.01,10.0275
1,A,P2,10.04,10.0275
2,A,P3,9.98,10.0275
3,A,P4,10.08,10.0275
4,B,P5,9.95,10.0100
5,B,P6,9.97,10.0100
6,B,P7,10.02,10.0100
7,B,P8,10.10,10.0100


Every part now has the average measurement of its own production line.

## 27. Deviation from Line Average

In [27]:
production["deviation_from_line_avg"] = (
    production["Measurement"]
    - production["line_avg"]
).round(4)

production

,Line,Part,Measurement,line_avg,deviation_from_line_avg
0,A,P1,10.01,10.0275,-0.0175
1,A,P2,10.04,10.0275,0.0125
2,A,P3,9.98,10.0275,-0.0475
3,A,P4,10.08,10.0275,0.0525
4,B,P5,9.95,10.0100,-0.0600
5,B,P6,9.97,10.0100,-0.0400
6,B,P7,10.02,10.0100,0.0100
7,B,P8,10.10,10.0100,0.0900


Positive deviation means the measurement is above its line average.

Negative deviation means it is below its line average.

## 28. Absolute Deviation

In [28]:
production["absolute_deviation"] = (
    production["deviation_from_line_avg"]
    .abs()
    .round(4)
)

production

,Line,Part,Measurement,line_avg,deviation_from_line_avg,absolute_deviation
0,A,P1,10.01,10.0275,-0.0175,0.0175
1,A,P2,10.04,10.0275,0.0125,0.0125
2,A,P3,9.98,10.0275,-0.0475,0.0475
3,A,P4,10.08,10.0275,0.0525,0.0525
4,B,P5,9.95,10.0100,-0.0600,0.0600
5,B,P6,9.97,10.0100,-0.0400,0.0400
6,B,P7,10.02,10.0100,0.0100,0.0100
7,B,P8,10.10,10.0100,0.0900,0.0900


Absolute deviation tells us the magnitude of the difference without considering direction.

## 29. Line-wise Ranking

In [29]:
production["line_rank"] = (
    production
    .groupby("Line")["Measurement"]
    .rank(
        ascending=False,
        method="dense"
    )
)

production.sort_values(["Line", "line_rank"])

,Line,Part,Measurement,line_avg,deviation_from_line_avg,absolute_deviation,line_rank
3,A,P4,10.08,10.0275,0.0525,0.0525,1.0
1,A,P2,10.04,10.0275,0.0125,0.0125,2.0
0,A,P1,10.01,10.0275,-0.0175,0.0175,3.0
2,A,P3,9.98,10.0275,-0.0475,0.0475,4.0
7,B,P8,10.10,10.0100,0.0900,0.0900,1.0
6,B,P7,10.02,10.0100,0.0100,0.0100,2.0
5,B,P6,9.97,10.0100,-0.0400,0.0400,3.0
4,B,P5,9.95,10.0100,-0.0600,0.0600,4.0


Rank 1 represents the highest measurement within each production line.

## 30. Above-Line-Average Flag

In [30]:
production["above_line_average"] = np.where(
    production["Measurement"] > production["line_avg"],
    "Yes",
    "No"
)

production

,Line,Part,Measurement,line_avg,deviation_from_line_avg,absolute_deviation,line_rank,above_line_average
0,A,P1,10.01,10.0275,-0.0175,0.0175,3.0,No
1,A,P2,10.04,10.0275,0.0125,0.0125,2.0,Yes
2,A,P3,9.98,10.0275,-0.0475,0.0475,4.0,No
3,A,P4,10.08,10.0275,0.0525,0.0525,1.0,Yes
4,B,P5,9.95,10.0100,-0.0600,0.0600,4.0,No
5,B,P6,9.97,10.0100,-0.0400,0.0400,3.0,No
6,B,P7,10.02,10.0100,0.0100,0.0100,2.0,Yes
7,B,P8,10.10,10.0100,0.0900,0.0900,1.0,Yes


# 31. QA Interpretation

A group benchmark and a specification limit are not the same thing.

For example, a part may be above its production-line average but still be completely within specification.

Therefore:

```text
Group benchmark
      +
Specification limits
      +
Process knowledge
      ↓
Better QA analysis
```

Do not automatically classify an unusual observation as defective.

# 32. Practical Analysis Pattern

This reusable pattern appears frequently in real-world analytics.

In [31]:
# Group benchmark
df = production.copy()

df["group_mean"] = (
    df.groupby("Line")["Measurement"]
    .transform("mean")
)

# Difference
df["difference"] = (
    df["Measurement"] - df["group_mean"]
)

# Percentage difference
df["percentage_difference"] = (
    df["difference"]
    / df["group_mean"]
    * 100
).round(2)

# Above average flag
df["above_average"] = np.where(
    df["Measurement"] > df["group_mean"],
    "Yes",
    "No"
)

df

,Line,Part,Measurement,line_avg,deviation_from_line_avg,absolute_deviation,line_rank,above_line_average,group_mean,difference,percentage_difference,above_average
0,A,P1,10.01,10.0275,-0.0175,0.0175,3.0,No,10.0275,-0.0175,-0.17,No
1,A,P2,10.04,10.0275,0.0125,0.0125,2.0,Yes,10.0275,0.0125,0.12,Yes
2,A,P3,9.98,10.0275,-0.0475,0.0475,4.0,No,10.0275,-0.0475,-0.47,No
3,A,P4,10.08,10.0275,0.0525,0.0525,1.0,Yes,10.0275,0.0525,0.52,Yes
4,B,P5,9.95,10.0100,-0.0600,0.0600,4.0,No,10.0100,-0.0600,-0.60,No
5,B,P6,9.97,10.0100,-0.0400,0.0400,3.0,No,10.0100,-0.0400,-0.40,No
6,B,P7,10.02,10.0100,0.0100,0.0100,2.0,Yes,10.0100,0.0100,0.10,Yes
7,B,P8,10.10,10.0100,0.0900,0.0900,1.0,Yes,10.0100,0.0900,0.90,Yes


# 33. Interview Questions

### Q1. What is the difference between `groupby().agg()` and `groupby().transform()`?

**Answer:**  
`agg()` produces a summarized result with fewer rows. `transform()` calculates a group-level operation but returns a result aligned with the original DataFrame rows.

### Q2. When would you use `transform()`?

When you need group statistics such as mean, median, total, maximum, or minimum attached to every original observation.

### Q3. What does `shift()` do?

It shifts values between rows and is commonly used to access previous or next observations.

### Q4. What does `cumsum()` do?

It calculates a cumulative or running sum.

### Q5. What is a rolling average?

An average calculated over a moving window of observations.

### Q6. Why must time-series data be sorted before using `shift()`?

Because `shift()` follows row order. Without chronological sorting, the previous row may not represent the previous time period.

### Q7. What is group-wise ranking?

Ranking observations separately within each group, for example:

```python
df.groupby("Line")["Measurement"].rank()
```

# 34. Lesson 12 Assignment

## Part A — Restaurant Analysis

Using `tips`:

1. Create `day_avg_bill` using `transform()`.
2. Create `difference_from_day_avg`.
3. Create `pct_diff_from_day_avg`.
4. Create `day_total_bill` using `transform("sum")`.
5. Create `pct_of_day_revenue`.
6. Rank transactions within each day using `day_rank`.
7. Find the top 3 transactions for each day.

### Suggested starting point

```python
assignment = tips.copy()
```

## Part B — Manufacturing Analysis

Using the `production` DataFrame:

Create:

1. `line_avg`
2. `line_median`
3. `deviation_from_line_avg`
4. `absolute_deviation`
5. `line_rank`
6. `above_line_average`

Then answer:

- Which part has the highest measurement within each production line?
- Which part deviates the most from its line average?
- Which line has the higher average measurement?

# 35. Final Lesson Summary

The key concepts from Lesson 12 are:

```text
groupby()
    ↓
Aggregation
    ↓
transform()
    ↓
Group vs Row Comparison
    ↓
rank()
    ↓
Group-wise Ranking
    ↓
shift()
    ↓
Previous / Next Comparison
    ↓
cumsum()
    ↓
Running Totals
    ↓
cumcount()
    ↓
Sequence Within Group
    ↓
rolling()
    ↓
Moving Averages
```

## Most Important Function

Whenever you think:

> **"I need the group statistic attached to every original row."**

Think:

```python
df.groupby("Group")["Value"].transform("mean")
```

### Key principle

> **Aggregation summarizes groups. Transformation preserves the original row structure.**

---

## End of Lesson 12

Complete the assignment in your Jupyter/Kaggle notebook before moving to the next lesson.